# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [16]:
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print("Total revenue:", total_revenue)
print("Total units:", total_units)
print("Number of rows:", len(df))

Total revenue: 8520.0
Total units: 783
Number of rows: 400


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [7]:
by_category = df.groupby('category', as_index=False)['revenue'].sum()
by_category['share_pct'] = (
    by_category['revenue'] / df['revenue'].sum() * 100
)

by_category = by_category.sort_values(
    by='revenue', ascending=False
)

by_category['revenue'] = by_category['revenue'].round(2)
by_category['share_pct'] = by_category['share_pct'].round(1)

print(by_category)

   category  revenue  share_pct
1      Food   4293.0       50.4
2     Merch   1771.5       20.8
0     Drink   1554.0       18.2
3  RainGear    901.5       10.6


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [8]:
by_vendor = df.groupby('vendor_id').agg(
    average_revenue=('revenue', 'mean'),
    order_count=('revenue', 'count')
).reset_index()

by_vendor = by_vendor.sort_values(
    by='average_revenue', ascending=False
)

by_vendor['average_revenue'] = by_vendor['average_revenue'].round(2)

print(by_vendor)

  vendor_id  average_revenue  order_count
0      V-01            22.60           94
3      V-18            21.75          108
1      V-05            20.58           93
2      V-10            20.31          105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [9]:
merch_revenue = df.loc[
    df['category'] == 'Merch', 'revenue'
].sum()

merch_share = merch_revenue / df['revenue'].sum() * 100

print("Merch revenue:", round(merch_revenue, 2))
print("Merch share of revenue:", round(merch_share, 1), "%")

Merch revenue: 1771.5
Merch share of revenue: 20.8 %


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [14]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
original_rows = len(df)
original_revenue = df['revenue'].sum()

joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

unmatched = joined.loc[
    joined['vendor_name'].isna(), 'vendor_id'
].unique()

print("Unmatched vendor IDs:", unmatched)

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor')

print("Original rows:", original_rows)
print("Joined rows:", len(joined))
print("Original revenue:", round(original_revenue, 2))
print("Joined revenue:", round(joined['revenue'].sum(), 2))

Unmatched vendor IDs: ['V-18']
Original rows: 400
Joined rows: 400
Original revenue: 8520.0
Joined revenue: 8520.0


**The unmatched vendor, and what I did about it:** V-18 and I labeled it unknown vendor and used a left join to retain all the order and avoid losing revenue from that vendor

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [12]:
pivot_report = pd.pivot_table(
    joined,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

pivot_report = pivot_report.round(2)
print(pivot_report)

category          Drink    Food   Merch  RainGear   Total
vendor_name                                              
Cav Merch North   502.5  1054.5   400.5     175.5  2133.0
Hoos Burgers      171.0  1338.0   373.5     241.5  2124.0
Rotunda Tacos     298.5   882.0   489.0     244.5  1914.0
Unknown vendor    582.0  1018.5   508.5     240.0  2349.0
Total            1554.0  4293.0  1771.5     901.5  8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [13]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a. Based on the demand reports I would tell the vendors to adjust their inventory based on the total and category specific revenues in order to plan for the next game. Food generated the most revenue at 4,293 dollars, accounting for 50.4 percent of the total revenue. Hoos Burgers had the highest average order revenue at 22.60 dollars across 94 orders. While revenue alone does not directly indicated future demand and overall profit, it can be helpful to see what is selling and where to determine how inventory can be optimized to optimize costs over the season.

b. I would consider question 5, with our unknown vendor, to be the least trustworthy from our study, because V-18 is not included in the vendor lookup table. Althought it accounts for 2,349 dollars in revenue across 108 orders, it doesn't make up for the fact that we don't know who we are dealing with. This makes it hard for us to make any vendor specific recommendations for next game and limits our ability to potentially make more revenue.